# Notebook 25 — DTI Integration

### Persistent Racial Disparities in U.S. Mortgage Approval

The most important new analysis: adds `debt_to_income_ratio` (present in your HMDA 2018+ files) 
to the DFL decomposition and within-lender fixed-effects specification.

**Addresses:** The #1 criticism across all 8 reviewers — DTI omission.

**Output files:**
- `data/processed/panel_with_dti.parquet` (used by NB27)
- `outputs/tables/table_25_dfl_with_dti.csv`
- `outputs/tables/table_25_dti_integration.csv`

**Runtime:** ~2 hours  **RAM:** ~6 GB peak


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

PROC       = Path("../data/processed"); PROC.mkdir(exist_ok=True)
RAW        = Path("../data")
TABS       = Path("../outputs/tables"); TABS.mkdir(exist_ok=True)
YEARS      = [2020, 2021, 2022, 2023, 2024]
BLACK_CODE = 3
WHITE_CODE = 5
CHUNK_SIZE = 200_000
MAX_PER    = 250   # per lender-race cap for FE

# ----------------------------------------------------------------
# DTI PARSER — converts HMDA string DTI to numeric midpoint
# ----------------------------------------------------------------
def parse_dti(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ("NA", "Exempt", "", "nan", "9999", "Exempted"):
        return np.nan
    range_map = {
        "<20%": 17.5, "20%-30%": 25.0, "30%-36%": 33.0,
        "36%-50%": 43.0, "50%-60%": 55.0, ">60%": 65.0,
    }
    if s in range_map:
        return range_map[s]
    # Single value like "37", "42%", ">60"
    try:
        v = float(s.replace("%","").replace(">","").replace("<","").strip())
        return v if 0 < v <= 100 else np.nan
    except (ValueError, TypeError):
        return np.nan

# Verify parser
tests = [("42", 42.0), ("50%-60%", 55.0), (">60%", 65.0),
         ("<20%", 17.5), ("NA", np.nan), ("Exempt", np.nan), ("36", 36.0)]
print("DTI parser verification:")
all_ok = True
for inp, expected in tests:
    result = parse_dti(inp)
    ok = (np.isnan(result) and np.isnan(expected)) or result == expected
    status = "✓" if ok else "✗"
    if not ok: all_ok = False
    print(f"  {status}  parse_dti({repr(inp):15s}) = {result}")
print(f"Parser: {'ALL CORRECT' if all_ok else 'ERRORS FOUND'}")


In [ ]:
# ----------------------------------------------------------------
# STEP 1: Load raw HMDA files, extract DTI + AUS, save parquet
# ----------------------------------------------------------------
print("="*65)
print("STEP 1: Loading raw HMDA files (30-40 min for all years)")
print("="*65)

RAW_COLS = [
    "activity_year", "lei", "action_taken",
    "applicant_race_1", "income", "loan_amount",
    "property_value", "combined_loan_to_value_ratio",
    "loan_purpose", "debt_to_income_ratio", "aus_1",
]

raw_dfs = []

for yr in YEARS:
    found = None
    for fname in [
        f"hmda_{yr}.csv",
        f"hmda_{yr}_nationwide_all-records_labels.csv",
        f"{yr}_public_lar_csv.csv",
    ]:
        if (RAW / fname).exists():
            found = RAW / fname
            break
    if found is None:
        print(f"  {yr}: NOT FOUND — skip")
        continue

    print(f"  {yr}: {found.name}")
    year_chunks = []
    reader = pd.read_csv(
        found, chunksize=CHUNK_SIZE,
        usecols=lambda c: c in RAW_COLS,
        engine="python", on_bad_lines="skip", dtype=str,
    )
    for chunk in reader:
        race = pd.to_numeric(chunk["applicant_race_1"], errors="coerce")
        act  = pd.to_numeric(chunk["action_taken"],     errors="coerce")
        mask = race.isin([BLACK_CODE, WHITE_CODE]) & act.isin([1, 3])
        chunk = chunk[mask].copy()
        if len(chunk):
            year_chunks.append(chunk)

    if not year_chunks:
        print(f"    No matching rows — skip")
        continue

    df_yr = pd.concat(year_chunks, ignore_index=True)
    df_yr["year"]        = yr
    df_yr["approved"]    = (pd.to_numeric(df_yr["action_taken"], errors="coerce") == 1).astype(int)
    df_yr["black"]       = (pd.to_numeric(df_yr["applicant_race_1"], errors="coerce") == BLACK_CODE).astype(int)
    df_yr["income_n"]    = pd.to_numeric(df_yr["income"],                          errors="coerce")
    df_yr["loan_n"]      = pd.to_numeric(df_yr["loan_amount"],                     errors="coerce")
    df_yr["prop_n"]      = pd.to_numeric(df_yr.get("property_value"),              errors="coerce")
    df_yr["ltv_n"]       = pd.to_numeric(df_yr["combined_loan_to_value_ratio"],    errors="coerce")
    df_yr["dti_numeric"] = df_yr["debt_to_income_ratio"].apply(parse_dti)
    df_yr["has_dti"]     = df_yr["dti_numeric"].notna()

    keep_cols = ["lei","year","approved","black","income_n","loan_n",
                 "prop_n","ltv_n","dti_numeric","has_dti","aus_1"]
    raw_dfs.append(df_yr[[c for c in keep_cols if c in df_yr.columns]])

    dti_pct = df_yr["has_dti"].mean() * 100
    print(f"    Rows kept: {len(df_yr):,}  |  DTI available: {dti_pct:.1f}%")

if not raw_dfs:
    raise RuntimeError("No raw HMDA files found in data/. Check filenames.")

df_all = pd.concat(raw_dfs, ignore_index=True)
print(f"\nCombined: {len(df_all):,} rows across {df_all['year'].nunique()} years")
print(f"DTI available: {df_all['has_dti'].mean()*100:.1f}% overall")
print(f"  Black: {df_all[df_all['black']==1]['has_dti'].mean()*100:.1f}%")
print(f"  White: {df_all[df_all['black']==0]['has_dti'].mean()*100:.1f}%")

df_all.to_parquet(PROC / "panel_with_dti.parquet", index=False)
print(f"\nSaved: data/processed/panel_with_dti.parquet")


In [ ]:
# ----------------------------------------------------------------
# STEP 2: DFL Decomposition — Without DTI vs With DTI
#
# IMPORTANT: Both specs use the SAME SAMPLE (obs with DTI available).
# This is the correct design for a fair comparison — if we used the
# full sample for "Without DTI" and the smaller sample for "With DTI",
# any change in the gap could reflect sample composition, not DTI.
# ----------------------------------------------------------------
print("="*65)
print("STEP 2: DFL DECOMPOSITION — Without DTI vs With DTI")
print("(Both specs estimated on DTI-available subsample for fair comparison)")
print("="*65)

# Restrict to obs with DTI available AND all other key vars
df_dti = df_all.dropna(subset=["income_n","loan_n","ltv_n","dti_numeric"]).copy()

# Check that prop_n is available for at least some years
prop_avail = df_dti["prop_n"].notna().mean()
print(f"property_value available in DTI sample: {prop_avail*100:.1f}%")
if prop_avail < 0.5:
    print("WARNING: property_value missing >50% — dropping from DFL features")
    use_prop = False
else:
    df_dti = df_dti.dropna(subset=["prop_n"])
    use_prop = True

for col in ["income_n","loan_n","prop_n"]:
    if col in df_dti.columns:
        df_dti[col+"_log"] = np.log1p(np.maximum(df_dti[col], 0))

results_dfl = []

for yr in YEARS:
    dfyr = df_dti[df_dti["year"] == yr].copy()
    if len(dfyr) < 500:
        print(f"  {yr}: insufficient obs ({len(dfyr)}) — skip")
        continue

    b_all = dfyr[dfyr["black"]==1]
    w_all = dfyr[dfyr["black"]==0]
    raw_gap = (w_all["approved"].mean() - b_all["approved"].mean()) * 100

    base_feats = ["income_n_log","ltv_n"]
    if use_prop and "prop_n_log" in dfyr.columns:
        base_feats.append("prop_n_log")
    if "loan_n_log" in dfyr.columns:
        base_feats.append("loan_n_log")

    for label, features in [
        ("Without DTI", base_feats),
        ("With DTI",    base_feats + ["dti_numeric"]),
    ]:
        df_sub = dfyr.dropna(subset=features)
        b_sub  = df_sub[df_sub["black"]==1]
        w_sub  = df_sub[df_sub["black"]==0]
        if len(b_sub) < 100 or len(w_sub) < 100:
            continue

        # Sample 1:5 ratio, max 100k White
        n_b = min(len(b_sub), 20_000)
        n_w = min(len(w_sub), 100_000)
        b_s = b_sub.sample(n_b, random_state=42)
        w_s = w_sub.sample(n_w, random_state=42)
        df_s = pd.concat([b_s, w_s], ignore_index=True)

        # Propensity score: P(White=1|X) — y=1 for White
        X = df_s[features].values
        y = (df_s["black"]==0).astype(int).values
        sc = StandardScaler()
        X_sc = sc.fit_transform(X)
        lr = LogisticRegression(max_iter=600, C=1.0, random_state=42)
        lr.fit(X_sc, y)

        # DFL weight for White: w = P(Black|X) / P(White|X)
        p_white = lr.predict_proba(sc.transform(w_s[features].values))[:,1]
        p_black = 1.0 - p_white
        weights = p_black / (p_white + 1e-9)
        weights = np.clip(weights, 0, np.percentile(weights, 95))

        rw_white = np.average(w_s["approved"].values, weights=weights)
        b_rate   = b_s["approved"].mean()
        raw_g    = (w_s["approved"].mean() - b_rate) * 100
        unexpl   = (rw_white - b_rate) * 100
        pct_unexpl = unexpl / raw_g * 100 if raw_g != 0 else 0

        results_dfl.append({
            "Year": yr, "Spec": label,
            "Raw_Gap_pp": round(raw_g, 3),
            "Unexplained_pp": round(unexpl, 3),
            "Explained_pp": round(raw_g - unexpl, 3),
            "Pct_Unexplained": round(pct_unexpl, 1),
            "N_Black": n_b, "N_White": n_w,
        })
        print(f"  {yr} {label:12s}: Raw={raw_g:.2f}pp  "
              f"Unexplained={unexpl:.2f}pp ({pct_unexpl:.1f}%)")

df_dfl = pd.DataFrame(results_dfl)
df_dfl.to_csv(TABS / "table_25_dfl_with_dti.csv", index=False)
print("\nSaved: table_25_dfl_with_dti.csv")
print()
print("KEY RESULT — How much does DTI reduce the unexplained fraction?")
for yr in YEARS:
    sub = df_dfl[df_dfl["Year"]==yr]
    if len(sub) == 2:
        wo = sub[sub["Spec"]=="Without DTI"]["Pct_Unexplained"].values[0]
        wi = sub[sub["Spec"]=="With DTI"]["Pct_Unexplained"].values[0]
        print(f"  {yr}: Without DTI={wo:.1f}%  With DTI={wi:.1f}%  Delta={wi-wo:+.1f}pp")


In [ ]:
# ----------------------------------------------------------------
# STEP 3: Within-Lender FE — Without DTI vs With DTI
# ----------------------------------------------------------------
print("="*65)
print("STEP 3: WITHIN-LENDER FE — Without DTI vs With DTI")
print("="*65)

def run_fe(df_input, controls, label, yr):
    """
    Frisch-Waugh-Lovell within-lender OLS with lender-clustered SE.
    controls: list of column names (must include 'black')
    Returns dict of results or None if insufficient data.
    """
    df = df_input.copy().dropna(subset=["approved","lei"] + controls)

    # Cap: max MAX_PER Black + MAX_PER White per lender
    def cap_lender(g):
        b = g[g["black"]==1]; w = g[g["black"]==0]
        return pd.concat([
            b.sample(min(len(b), MAX_PER), random_state=42),
            w.sample(min(len(w), MAX_PER), random_state=42),
        ])
    df = df.groupby("lei", group_keys=False).apply(cap_lender)

    # FWL within-transformation: demean by lender
    lender_means = df.groupby("lei")[["approved"] + controls].transform("mean")
    df_dm = pd.DataFrame()
    df_dm["approved_dm"] = df["approved"] - lender_means["approved"]
    for c in controls:
        df_dm[c+"_dm"] = df[c] - lender_means[c]
    df_dm["lei"] = df["lei"].values
    df_dm = df_dm.dropna()

    Xc = [c+"_dm" for c in controls]
    X  = df_dm[Xc].values
    y  = df_dm["approved_dm"].values
    lei_arr = df_dm["lei"].values

    Xf = np.column_stack([np.ones(len(X)), X])
    coef, _, _, _ = np.linalg.lstsq(Xf, y, rcond=None)
    resid = y - Xf @ coef
    ul = np.unique(lei_arr); G = len(ul); n = len(y); k = Xf.shape[1]
    if G < 2: return None

    # Clustered sandwich SE (HC)
    adj = (G / (G-1)) * ((n-1) / (n-k))
    try:    bread = np.linalg.inv(Xf.T @ Xf)
    except: bread = np.linalg.pinv(Xf.T @ Xf)
    meat = np.zeros((k, k))
    for lend in ul:
        idx = (lei_arr == lend)
        score = Xf[idx].T @ resid[idx]
        meat += np.outer(score, score)
    vcov = adj * bread @ meat @ bread
    se_arr = np.sqrt(np.maximum(np.diag(vcov), 0))

    col_names = ["const"] + Xc
    bi = col_names.index("black_dm")
    beta = coef[bi] * 100
    bse  = se_arr[bi] * 100
    ts   = beta / bse if bse > 0 else 0
    pval = 2 * (1 - stats.t.cdf(abs(ts), df=G-1))
    sig  = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."

    return {"Year": yr, "Spec": label, "Beta_pp": round(beta,3),
            "SE": round(bse,3), "T_stat": round(ts,3),
            "P_value": round(pval,4), "Sig": sig,
            "N_obs": n, "N_lenders": G}

# Base controls (always available)
base_controls = ["black","income_n","loan_n","ltv_n"]

results_fe = []
for yr in YEARS:
    dfyr = df_all[df_all["year"]==yr].dropna(subset=["income_n","loan_n","ltv_n"]).copy()
    if len(dfyr) < 500: continue
    print(f"  {yr}:")

    # Spec 1: Without DTI
    r1 = run_fe(dfyr, base_controls, "Without DTI", yr)
    if r1:
        results_fe.append(r1)
        print(f"    Without DTI: {r1['Beta_pp']:+.3f}pp  {r1['Sig']}")

    # Spec 2: With DTI (only obs with DTI available)
    dfyr_dti = dfyr.dropna(subset=["dti_numeric"])
    if len(dfyr_dti) > 500:
        r2 = run_fe(dfyr_dti, base_controls + ["dti_numeric"], "With DTI", yr)
        if r2:
            results_fe.append(r2)
            print(f"    With DTI:    {r2['Beta_pp']:+.3f}pp  {r2['Sig']}"
                  f"  (N={r2['N_obs']:,}  Lenders={r2['N_lenders']:,})")

df_fe = pd.DataFrame(results_fe)
df_fe.to_csv(TABS / "table_25_dti_integration.csv", index=False)
print("\nSaved: table_25_dti_integration.csv")

print()
print("SUMMARY: How much does adding DTI change the within-lender gap?")
for yr in YEARS:
    sub = df_fe[df_fe["Year"]==yr]
    if len(sub) >= 2:
        wo = sub[sub["Spec"]=="Without DTI"]["Beta_pp"].values[0]
        wi = sub[sub["Spec"]=="With DTI"]["Beta_pp"].values[0]
        print(f"  {yr}: Without={wo:+.3f}pp  With={wi:+.3f}pp  Delta={wi-wo:+.3f}pp")

print()
print("NB25 COMPLETE")
print("Outputs:")
print("  data/processed/panel_with_dti.parquet  (used by NB27)")
print("  outputs/tables/table_25_dfl_with_dti.csv")
print("  outputs/tables/table_25_dti_integration.csv")
